# Assignment 7: Transfer learning with AlexNet, VGG-16, ResNet50, EfficientNet-B0
All four pretrained ImageNet models come from `torchvision` (Keras has no AlexNet). The backbone is frozen, only the new classifier head is trained, on a CIFAR-10 subset resized to 224x224.

In [ ]:
import time, numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn, torchvision
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import accuracy_score, f1_score

torch.manual_seed(42); np.random.seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

In [ ]:
tf_ = transforms.Compose([transforms.Resize(224), transforms.ToTensor(),
                          transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
train_full = datasets.CIFAR10("./data", train=True, download=True, transform=tf_)
test_full = datasets.CIFAR10("./data", train=False, download=True, transform=tf_)

N_TRAIN, N_TEST = 5000, 1000   # subset for speed; raise if you have time
tr_idx = np.random.permutation(len(train_full))[:N_TRAIN]; te_idx = np.random.permutation(len(test_full))[:N_TEST]
train_dl = DataLoader(Subset(train_full, tr_idx), batch_size=64, shuffle=True, num_workers=2)
test_dl = DataLoader(Subset(test_full, te_idx), batch_size=128, num_workers=2)
classes = train_full.classes

In [ ]:
NUM_CLASSES = 10

def get_model(name):
    if name == "AlexNet":
        m = models.alexnet(weights=models.AlexNet_Weights.DEFAULT)
        for p in m.parameters(): p.requires_grad = False
        m.classifier[6] = nn.Linear(4096, NUM_CLASSES)
    elif name == "VGG16":
        m = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
        for p in m.parameters(): p.requires_grad = False
        m.classifier[6] = nn.Linear(4096, NUM_CLASSES)
    elif name == "ResNet50":
        m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        for p in m.parameters(): p.requires_grad = False
        m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    elif name == "EfficientNetB0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        for p in m.parameters(): p.requires_grad = False
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, NUM_CLASSES)
    return m.to(device)

def run(name, epochs=3, lr=1e-3):
    m = get_model(name)
    params = [p for p in m.parameters() if p.requires_grad]
    opt = torch.optim.Adam(params, lr=lr); crit = nn.CrossEntropyLoss()
    t0 = time.time(); hist = []
    for ep in range(epochs):
        m.train(); tot = 0
        for x, y in train_dl:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(); loss = crit(m(x), y); loss.backward(); opt.step(); tot += loss.item() * len(x)
        hist.append(tot / len(train_dl.dataset))
        print(f"{name} epoch {ep+1}/{epochs} loss {hist[-1]:.4f}")
    train_time = time.time() - t0
    m.eval(); yt, yp = [], []
    with torch.no_grad():
        for x, y in test_dl:
            yp += m(x.to(device)).argmax(1).cpu().tolist(); yt += y.tolist()
    return {"Model": name, "Accuracy": accuracy_score(yt, yp), "Macro F1": f1_score(yt, yp, average="macro"),
            "Trainable params": sum(p.numel() for p in params), "Total params": sum(p.numel() for p in m.parameters()),
            "Train time (s)": train_time}, hist

results, losses = [], {}
for name in ["AlexNet", "VGG16", "ResNet50", "EfficientNetB0"]:
    r, h = run(name); results.append(r); losses[name] = h
    torch.cuda.empty_cache() if device == "cuda" else None
res = pd.DataFrame(results).set_index("Model")
res.round(4)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
res["Accuracy"].plot(kind="bar", ax=ax[0], title="Test accuracy", ylim=(0, 1))
res["Train time (s)"].plot(kind="bar", ax=ax[1], title="Training time (s)")
for k, h in losses.items(): ax[2].plot(range(1, len(h)+1), h, marker="o", label=k)
ax[2].set_title("Training loss"); ax[2].set_xlabel("epoch"); ax[2].legend()
plt.tight_layout(); plt.show()

## Comparison notes
| Model | Idea | Size | Typical behaviour |
|---|---|---|---|
| AlexNet | 5 conv + 3 FC, ReLU, dropout | ~61M params | fastest, lowest accuracy |
| VGG-16 | stacks of 3x3 convs | ~138M params | strong features but very heavy and slow |
| ResNet50 | residual (skip) connections | ~25M params | strong accuracy, good speed |
| EfficientNet-B0 | compound scaling | ~5M params | best accuracy per parameter |

Exact numbers depend on subset size and epochs; read the table above.